# Graph use and path finding

In this example, we grab a pre-existing model and do the following:

* Re-compute fields in the network
* Manipulate the graph in memory
* Compute paths

## Running on Google Colab

Press here to open this notebook in Google Colab <a href="https://colab.research.google.com/github/outerl/AequilibraE-demo/blob/main/path_finding_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

Once in Colab, uncomment the cell below and run it.

**It will restart your Python environment to load all packages correctly**, so it is not possible just run all cells

In [2]:
# !apt-get update && apt-get install libsqlite3-mod-spatialite
# !apt-get install -y libspatialite-dev
# !pip install numpy --upgrade
# !pip install aequilibrae matplotlib
# exit()

# Before we begin
Let's make a copy of the model so we can make the changes we want without overwriting the original model

If you have are running this notebook on Google Colab and want to save your model into your Google Drive, you should:

1. Un-comment and run the two cells below
2. Accept the terms and conditions when asking to connect Colab to Google Drive.
3. Skip the following cell, where we copy the model around

In [3]:
# #Just in case you wabt to use Google colab
# from google.colab import drive
# drive.mount("/content/gdrive")

In [4]:
# !wget https://github.com/outerl/AequilibraE-demo/releases/download/Freeworld/LongAn_base_model.zip
# import zipfile
# with zipfile.ZipFile('LongAn_base_model.zip',"r") as zip_ref:
#     zip_ref.extractall(".")

# # You probably want to move this into your google drive, but we will leave in iun the temporary folder of the VM running the model
# model_path = "./LongAn_base_model"

In [5]:
model_path = r"C:\Users\Pedro\Downloads\simplifier\goldcoast"

If not using Colab, you will need to download the [MODEL](https://github.com/outerl/AequilibraE-demo/releases/download/Freeworld/LongAn_base_model.zip), unzip it and set the folders below accordingly

## Let's do some modeling

In [6]:
# Imports
from pathlib import Path
from aequilibrae import Project
import geopandas as gpd

In [7]:
# We open a project
project = Project.from_path(Path(model_path))

## Model stats

In [8]:
print(f"Links: {project.network.count_links():,}")
print(f"Nodes: {project.network.count_nodes():,}")
print(f"Zones: {project.network.count_centroids():,}")

Links: 11,140
Nodes: 4,783
Zones: 1,068


Let's see which fields we have in our links layer

In [9]:
project.network.links.fields.all_fields()

['a_node',
 'b_node',
 'capacity',
 'direction',
 'distance',
 'geometry',
 'link_id',
 'link_type',
 'modes',
 'name',
 'speed',
 'travel_time']

## Computes Graph



In [10]:
%%time
project.network.build_graphs(modes=["c"])

CPU times: total: 109 ms
Wall time: 117 ms


In [11]:
graph = project.network.graphs["c"]

In [12]:
graph.available_skims()

['distance',
 'name',
 'speed',
 'travel_time',
 'capacity',
 'modes',
 '__supernet_id__',
 '__compressed_id__']

### We add a field for the generalized cost, which is a combination of distance and travel time. You can use any formula you want here, but we will just use a simple linear combination of the two.

In [13]:
graph.network = graph.network.assign(generalized_cost_ab=graph.network["distance"] * 0.005 + graph.network.travel_time_ab* 0.5)
graph.network = graph.network.assign(generalized_cost_ba=graph.network["distance"] * 0.005 + graph.network.travel_time_ba* 0.5)

graph.prepare_graph(graph.centroids)

# Let's use generalize cost for path finding
graph.set_graph("generalized_cost")

In [14]:
graph.available_skims()

['distance',
 'name',
 'speed',
 'travel_time',
 'capacity',
 'modes',
 'generalized_cost',
 '__supernet_id__',
 '__compressed_id__']

In [15]:
graph.set_skimming(["distance", "travel_time"])

In [16]:
display(graph.network)

,link_id,a_node,b_node,direction,distance,name,speed_ab,speed_ba,travel_time_ab,travel_time_ba,capacity_ab,capacity_ba,modes,id,generalized_cost_ab,generalized_cost_ba
0,1,1,1371,1,297.802333,NaN,NaN,NaN,0.327,NaN,900.0,NaN,bctw,NaN,1.652512,NaN
1,2,2,2012,1,257.996134,NaN,NaN,NaN,0.173,NaN,1600.0,NaN,bctw,NaN,1.376481,NaN
2,3,3,2402,1,202.356800,NaN,NaN,NaN,0.200,NaN,1100.0,NaN,bctw,NaN,1.111784,NaN
3,4,4,1875,1,156.384926,NaN,NaN,NaN,0.128,NaN,800.0,NaN,bctw,NaN,0.845925,NaN
4,5,5,1880,1,202.999207,NaN,NaN,NaN,0.171,NaN,800.0,NaN,bctw,NaN,1.100496,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11135,11136,4806,1495,1,93.652101,NaN,NaN,NaN,0.120,NaN,600.0,NaN,bctw,NaN,0.528261,NaN
11136,11137,4806,3606,1,91.541966,NaN,NaN,NaN,0.120,NaN,600.0,NaN,bctw,NaN,0.517710,NaN
11137,11138,4806,415,1,231.804919,NaN,NaN,NaN,0.575,NaN,1800.0,NaN,bctw,NaN,1.446525,NaN
11138,11139,4807,1433,1,54.513444,NaN,NaN,NaN,0.060,NaN,400.0,NaN,bctw,NaN,0.302567,NaN


In [17]:
display(graph.graph)

,link_id,a_node,b_node,direction,id,distance,name,speed,travel_time,capacity,modes,generalized_cost,__supernet_id__,__compressed_id__
0,1,0,1370,1,0,297.802333,NaN,NaN,0.327,900.0,bctw,1.652512,0,0
1,2,1,2011,1,1,257.996134,NaN,NaN,0.173,1600.0,bctw,1.376481,1,1
2,3,2,2386,1,2,202.356800,NaN,NaN,0.200,1100.0,bctw,1.111784,2,2
3,4,3,1874,1,3,156.384926,NaN,NaN,0.128,800.0,bctw,0.845925,3,3
4,5,4,1879,1,4,202.999207,NaN,NaN,0.171,800.0,bctw,1.100496,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11135,11138,4781,414,1,11135,231.804919,NaN,NaN,0.575,1800.0,bctw,1.446525,11137,10957
11136,11136,4781,1494,1,11136,93.652101,NaN,NaN,0.120,600.0,bctw,0.528261,11135,10958
11137,11137,4781,3585,1,11137,91.541966,NaN,NaN,0.120,600.0,bctw,0.517710,11136,10959
11138,11139,4782,1432,1,11138,54.513444,NaN,NaN,0.060,400.0,bctw,0.302567,11138,10960


In [18]:
# graph.exclude_links([1, 2, 3])

## Path finding

In [38]:
res = graph.compute_path(2, 1432)

In [22]:
res.path

array([    2,  4133,  4132,  3686,  9122,  9119,  9125,  8049,  4269,
        7997,  4515,  1837,  4260,  9041,  1574,  9037,  1832,  6793,
        1818,  5286,  1807,  1795,  1791,  1788,  1782,  1779,  1772,
        1744,  1709,  6917,  1702,  4280,  8463,  1692,  1688,  6926,
        1555,  5561,  1537,  5253,  1523,  1520,  5246,  1513,  1454,
        6938,  1441,  6935,  1437,  6929,  1435,  1426,  6934,  1429,
        1422,  4228,  4464,  1399,  9215,  9417,  5447,  4336,  5186,
        5178,  5825,  4470,  6944,  9760,  1268,  1258,  5822,  7958,
        5165,  6526,  1231, 11108,  6957,  1228,  1203,  9757,  1194,
        9755,  9933,  9935,  9937,  9946,  1196,  9942,  9940,  5508,
        1206,  5514,  1216,  5377, 10073,  3423,  5133,  3420,  3406,
        6985,  5120,  3403, 11111,  5115,  8502,  8879, 10111,  3397,
        9133, 10117, 10115,  4178,  9810,  9812,  9819,  3373,  9825,
        9827,  3344,  5081,  7433,  3341,  8885,  3332,  3314,  9806,
        3317,  9803,

In [23]:
res.path_link_directions

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [24]:
res.path_nodes

array([   2, 2012, 2011, 1855, 3909, 3908, 3910, 3471, 2053, 3446, 2155,
       1288, 2050, 3873, 1204, 3872, 1286, 3004, 1282, 2417, 1278, 1275,
       1274, 1273, 1271, 1270, 1268, 1259, 1246, 3057, 1244, 2057, 3632,
       1241, 1240, 3061, 1199, 2519, 1194, 2406, 1190, 1189, 2403, 1187,
       1170, 3065, 1167, 3064, 1166, 3062, 1165, 1162, 3063, 1163, 1161,
       2041, 2137, 1153, 3944, 4034, 2478, 2093, 2381, 2378, 2613, 2139,
       3068, 4190, 1110, 1107, 2612, 3431, 2374, 2911, 1099, 4795, 3073,
       1098, 1091, 4188, 1088, 4187, 4270, 4271, 4272, 4276, 1089, 4274,
       4273, 2500, 1092, 2502, 1095, 2449, 4332, 1772, 2364, 1771, 1766,
       3086, 2359, 1765, 4796, 2358, 3644, 3787, 4352, 1764, 3914, 4354,
       4353, 2026, 4214, 4215, 4218, 1757, 4220, 4221, 1750, 2347, 3256,
       1749, 3790, 1746, 1741, 4212, 1742, 4211, 3785, 4210, 1736, 4604,
       4605, 1734, 4603, 4599, 4602, 2620, 1723, 2586, 3788, 1724, 2623,
       1726, 1712, 2337, 4540, 1710, 4539, 1709, 17

In [25]:
res.milepost

array([  0.        ,   1.37648067,   5.85208604,   6.96510374,
        10.08020943,  11.01008025,  12.29297518,  12.90340239,
        15.77851151,  19.2274251 ,  20.14939935,  21.0686995 ,
        21.74512064,  23.4668635 ,  26.00232186,  30.12192109,
        31.32158235,  33.48948838,  35.76669588,  38.3856588 ,
        38.82263588,  40.21117708,  41.4925282 ,  42.69404006,
        43.60671762,  45.26546831,  47.06573405,  47.43492653,
        48.98897381,  50.14540187,  52.63690901,  53.20234363,
        54.10751223,  54.73832091,  55.86157425,  56.70108917,
        57.54402209,  58.12207358,  58.6788027 ,  60.36322498,
        61.47472646,  63.13143823,  64.26034843,  65.36134121,
        66.47398592,  68.16430486,  68.68953634,  69.76743679,
        71.27786278,  73.5460542 ,  75.9873047 ,  77.78230922,
        79.6887976 ,  80.20299003,  80.75935952,  81.53636224,
        82.2345522 ,  82.71586643,  82.91724605,  83.17165447,
        83.60031251,  84.55700838,  85.86652246,  86.40

We can save this matrix to disk and add to the model

In [ ]:
links = project.network.links.data

In [39]:
path = links[links.link_id.isin(res.path)]

path.explore(tiles="CartoDB positron", style_kwds={"weight": 5})

In [37]:
res.update_trace(3)
path = links[links.link_id.isin(res.path)]

path.explore(tiles="CartoDB positron", style_kwds={"weight": 5})